# What's new in Apache Iceberg 1.12 and what works in this lab

The lab runs **Iceberg 1.12.0** in Spark 4.1.3, but it also contains older Iceberg code: Trino 483 bundles Iceberg 1.11.0 and the Iceberg REST catalog inside Hive Metastore 4.2.1 embeds 1.9.1.
A new Iceberg feature therefore only works if *every* component it touches understands it. This notebook **tests the stack live**, so you get the real answer for the version you are running,
and points to the notebook that teaches each feature that does work.

Source of the feature list: the [1.12.0 release notes](https://iceberg.apache.org/releases/) (released 2026-09-29) and the 1.12.0 documentation.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from lab_utils import get_spark, try_sql, CATALOG

spark = get_spark("iceberg-12-overview")
NS = f"{CATALOG}.iceberg12"
_ = spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")
spark.sql(f"DROP TABLE IF EXISTS {NS}.probe_version")
spark.sql(f"CREATE TABLE {NS}.probe_version (id INT) USING iceberg")
spark.sql(f"INSERT INTO {NS}.probe_version VALUES (1)")
info = spark.sql(f"SELECT summary FROM {NS}.probe_version.snapshots LIMIT 1").collect()[0][0]
print("Spark          :", spark.version)
print("Iceberg runtime:", info.get("iceberg-version", "?"))

## Feature matrix (checked in this lab)
| Iceberg 1.12 feature | Status here | Where |
|---|---|---|
| Hilbert-curve clustering in `rewrite_data_files` (Spark 4.1) | works | `01_hilbert_clustering` |
| Format v3 tables and the `VARIANT` type on the Iceberg REST catalog; Spark 4.1 and Trino 483 read them | works | `02_variant_semi_structured` |
| Variant shredding (`write.parquet.shred-variants`) | works for reads and writes; **DELETE/lineage reads failed** until vectorization is disabled on the table | `02_variant_semi_structured` |
| v3 deletion vectors (Puffin) and row lineage (`_row_id`, `_last_updated_sequence_number`), Trino reads them | works | `03_deletion_vectors_and_row_lineage` |
| `remove-dangling-deletes` option of `rewrite_data_files` | works | `03_deletion_vectors_and_row_lineage` |
| `use-merge-append-for-streaming` write option | works | `04_streaming_and_parquet_tuning` |
| Per-column Parquet dictionary encoding | works | `04_streaming_and_parquet_tuning` |
| Parquet bloom filters incl. adaptive sizing | **not verified**: no bloom filter was found in files written through this stack | |
| `geometry` / `geography` types | **not supported here**: Spark 4.1 gates them behind a flag and the REST catalog server rejects them | probe below |
| Column default values (v3) through Spark SQL | **not supported** by Iceberg's Spark integration | probe below |
| `rewrite_manifests` with `sort_by` | works on Spark 4.1.3 (release notes list 3.5 and 4.0) | probe below |
| Session-level split size (`spark.sql.iceberg.read.split-size`) | accepted; effect not measured here | probe below |
| Flink 2.2/2.3, Kafka Connect, v4 spec groundwork, Rust/Python/C++ clients | not part of this lab | |

Run the probes to see the same facts on your own deployment:

In [ ]:
print("--- v3 tables / variant (REST catalog)")
try_sql(spark, f"CREATE TABLE IF NOT EXISTS {NS}.probe_v3 (id INT, v VARIANT) USING iceberg TBLPROPERTIES ('format-version'='3')", "create a format-v3 table with a VARIANT column")

print("--- geospatial types")
try:
    spark.conf.set("spark.sql.geospatial.enabled", "true")      # Spark 4.1 keeps the feature behind this flag
except Exception as exc:
    print("could not enable the Spark geospatial flag:", str(exc).splitlines()[0][:150])
try_sql(spark, f"CREATE TABLE IF NOT EXISTS {NS}.probe_geo (id INT, g GEOMETRY(4326)) USING iceberg TBLPROPERTIES ('format-version'='3')",
        "create a table with a GEOMETRY column")

print("--- column defaults through Spark SQL")
try_sql(spark, f"ALTER TABLE {NS}.probe_v3 ADD COLUMN region STRING DEFAULT 'eu'", "ADD COLUMN ... DEFAULT")

print("--- rewrite_manifests sort_by")
spark.sql(f"DROP TABLE IF EXISTS {NS}.probe_part")
spark.sql(f"CREATE TABLE {NS}.probe_part (id BIGINT, category STRING) USING iceberg PARTITIONED BY (category)")
spark.sql(f"INSERT INTO {NS}.probe_part SELECT id, concat('c', id % 3) FROM range(0, 300)")
try_sql(spark, f"CALL {CATALOG}.system.rewrite_manifests(table => 'iceberg12.probe_part', sort_by => array('category'))", "rewrite_manifests(sort_by => array('category'))")

print("--- session-level split size")
spark.conf.set("spark.sql.iceberg.read.split-size", "1048576")
try_sql(spark, f"SELECT count(*) FROM {NS}.probe_part", "read with spark.sql.iceberg.read.split-size = 1 MiB")

## Reading the results
* **FAILED is information, not a bug in the notebook.** The geometry error comes from the REST server (it reports HTTP 406 "geometry is not supported"): the catalog embedded in Hive Metastore 4.2.1
  is built on an older Iceberg and cannot store the new types. It will start working when the catalog side is upgraded.
* The lab's `docs/VERSIONS.md` lists the same constraint under "upgrade blockers": Trino 483 bundles Iceberg 1.11.0 and the REST catalog 1.9.1.
* If a probe that fails here prints `OK` in a later version of the lab, update the table above.

In [ ]:
CLEANUP = False   # set True to drop the probe tables
if CLEANUP:
    for t in ["probe_version", "probe_v3", "probe_geo", "probe_part"]:
        spark.sql(f"DROP TABLE IF EXISTS {NS}.{t}")